# Multi-Stage Translation Quality Analysis
## Comprehensive Statistical Analysis and Exploratory Data Analysis

**Research Question:** Does the number of spelling errors in the original text correlate with semantic degradation through a multi-stage translation pipeline (English → French → Hebrew → English)?

**Dataset:** 170 sentences with 4-20 intentional spelling errors (10 sentences per error count)

## 1. Setup and Data Loading

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, f_oneway
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Libraries loaded successfully!")

In [ ]:
# Load data
results_df = pd.read_csv('vector_similarity_results.csv')
averaged_df = pd.read_csv('vector_similarity_averaged.csv')

print(f"Individual results: {len(results_df)} rows")
print(f"Averaged groups: {len(averaged_df)} groups")
print("\nFirst few rows:")
results_df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Summary statistics
print("=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)
print("\nCosine Similarity:")
print(results_df['Cosine_Similarity'].describe())
print("\nError Count:")
print(results_df['Mistakes'].describe())

In [ ]:
# Distribution of similarity scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(results_df['Cosine_Similarity'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(results_df['Cosine_Similarity'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {results_df["Cosine_Similarity"].mean():.4f}')
axes[0].axvline(results_df['Cosine_Similarity'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {results_df["Cosine_Similarity"].median():.4f}')
axes[0].set_xlabel('Cosine Similarity', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=12, fontweight='bold')
axes[0].set_title('Distribution of Cosine Similarity Scores', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot
axes[1].boxplot(results_df['Cosine_Similarity'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_ylabel('Cosine Similarity', fontsize=12, fontweight='bold')
axes[1].set_title('Box Plot of Similarity Scores', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Identify outliers
Q1 = results_df['Cosine_Similarity'].quantile(0.25)
Q3 = results_df['Cosine_Similarity'].quantile(0.75)
IQR = Q3 - Q1
outliers = results_df[(results_df['Cosine_Similarity'] < Q1 - 1.5*IQR) | (results_df['Cosine_Similarity'] > Q3 + 1.5*IQR)]
print(f"\nNumber of outliers: {len(outliers)}")
if len(outliers) > 0:
    print("\nOutliers:")
    print(outliers[['Index', 'Mistakes', 'Cosine_Similarity']])

In [ ]:
# Similarity by error count
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Box plot by error count
results_df.boxplot(column='Cosine_Similarity', by='Mistakes', ax=axes[0, 0], patch_artist=True)
axes[0, 0].set_xlabel('Number of Mistakes', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Cosine Similarity', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Similarity Distribution by Error Count', fontsize=12, fontweight='bold')
axes[0, 0].get_figure().suptitle('')  # Remove auto-generated title

# 2. Violin plot
sns.violinplot(data=results_df, x='Mistakes', y='Cosine_Similarity', ax=axes[0, 1], color='lightblue')
axes[0, 1].set_xlabel('Number of Mistakes', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Cosine Similarity', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Violin Plot by Error Count', fontsize=12, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Scatter plot with trend line
axes[1, 0].scatter(results_df['Mistakes'], results_df['Cosine_Similarity'], 
                   alpha=0.6, s=50, color='steelblue', edgecolors='black', linewidth=0.5)
z = np.polyfit(results_df['Mistakes'], results_df['Cosine_Similarity'], 1)
p = np.poly1d(z)
x_trend = np.linspace(results_df['Mistakes'].min(), results_df['Mistakes'].max(), 100)
axes[1, 0].plot(x_trend, p(x_trend), "r--", linewidth=2, alpha=0.8, label=f'y = {z[0]:.5f}x + {z[1]:.4f}')
axes[1, 0].set_xlabel('Number of Mistakes', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Cosine Similarity', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Scatter Plot with Linear Trend', fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. Mean similarity by error count with error bars
axes[1, 1].errorbar(averaged_df['Mistakes'], averaged_df['Avg_Similarity'], 
                    yerr=averaged_df['Std_Dev'], fmt='o-', capsize=5, capthick=2,
                    markersize=8, linewidth=2, color='steelblue', ecolor='gray', label='Mean ± Std Dev')
axes[1, 1].set_xlabel('Number of Mistakes', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Average Cosine Similarity', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Average Similarity by Error Count', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Correlation Analysis

In [ ]:
# Calculate correlations
pearson_r, pearson_p = pearsonr(results_df['Mistakes'], results_df['Cosine_Similarity'])
spearman_r, spearman_p = spearmanr(results_df['Mistakes'], results_df['Cosine_Similarity'])

print("=" * 80)
print("CORRELATION ANALYSIS")
print("=" * 80)
print(f"\nPearson Correlation:")
print(f"  r = {pearson_r:.4f}")
print(f"  p-value = {pearson_p:.6f}")
print(f"  Interpretation: {'Statistically significant' if pearson_p < 0.05 else 'Not significant'} (α = 0.05)")

print(f"\nSpearman Correlation:")
print(f"  ρ = {spearman_r:.4f}")
print(f"  p-value = {spearman_p:.6f}")
print(f"  Interpretation: {'Statistically significant' if spearman_p < 0.05 else 'Not significant'} (α = 0.05)")

# R-squared
r_squared = pearson_r ** 2
print(f"\nR-squared:")
print(f"  r² = {r_squared:.4f}")
print(f"  Interpretation: {r_squared*100:.2f}% of variance in similarity is explained by error count")

In [ ]:
# Correlation heatmap
corr_data = results_df[['Mistakes', 'Cosine_Similarity']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_data, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=2, cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1, fmt='.4f')
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 4. Hypothesis Testing

In [ ]:
print("=" * 80)
print("HYPOTHESIS TESTING")
print("=" * 80)
print("\nNull Hypothesis (H0): ρ = 0 (no correlation)")
print("Alternative Hypothesis (H1): ρ ≠ 0 (correlation exists)")
print(f"Significance level: α = 0.05")

# Test statistic
n = len(results_df)
t_stat = pearson_r * np.sqrt((n-2)/(1-pearson_r**2))
df = n - 2

print(f"\nTest Statistic: t = {t_stat:.4f}")
print(f"Degrees of Freedom: {df}")
print(f"P-value: {pearson_p:.6f}")

if pearson_p < 0.05:
    print(f"\n✓ REJECT H0 (p < 0.05)")
    print(f"  Conclusion: There IS a statistically significant correlation")
    print(f"  Direction: {'NEGATIVE' if pearson_r < 0 else 'POSITIVE'} correlation")
else:
    print(f"\n✗ FAIL TO REJECT H0 (p ≥ 0.05)")
    print(f"  Conclusion: No statistically significant correlation")

# Confidence interval for correlation
z = 0.5 * np.log((1 + pearson_r) / (1 - pearson_r))
se = 1 / np.sqrt(n - 3)
z_crit = stats.norm.ppf(0.975)
z_lower = z - z_crit * se
z_upper = z + z_crit * se
r_lower = (np.exp(2 * z_lower) - 1) / (np.exp(2 * z_lower) + 1)
r_upper = (np.exp(2 * z_upper) - 1) / (np.exp(2 * z_upper) + 1)

print(f"\n95% Confidence Interval for r: [{r_lower:.4f}, {r_upper:.4f}]")
print(f"We are 95% confident the true correlation lies in this range")

## 5. ANOVA Analysis

In [ ]:
# Prepare groups for ANOVA
groups = []
for mistake_count in sorted(results_df['Mistakes'].unique()):
    group_data = results_df[results_df['Mistakes'] == mistake_count]['Cosine_Similarity'].values
    groups.append(group_data)

# Perform ANOVA
f_stat, p_value = f_oneway(*groups)

print("=" * 80)
print("ANOVA (Analysis of Variance)")
print("=" * 80)
print("\nNull Hypothesis (H0): All error count groups have equal mean similarity")
print("Alternative Hypothesis (H1): At least one group has different mean")

print(f"\nF-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.6f}")

if p_value < 0.05:
    print(f"\n✓ REJECT H0 (p < 0.05)")
    print(f"  Conclusion: At least one group has significantly different mean similarity")
else:
    print(f"\n✗ FAIL TO REJECT H0 (p ≥ 0.05)")
    print(f"  Conclusion: No significant differences between groups")

# Effect size (eta-squared)
grand_mean = results_df['Cosine_Similarity'].mean()
ss_between = sum([len(g) * (np.mean(g) - grand_mean)**2 for g in groups])
ss_total = sum([(x - grand_mean)**2 for x in results_df['Cosine_Similarity']])
eta_squared = ss_between / ss_total

print(f"\nEta-squared (η²): {eta_squared:.4f}")
print(f"Interpretation: {eta_squared*100:.2f}% of variance explained by error count groups")

## 6. Effect Size Analysis

In [ ]:
# Compare extreme groups (4 vs 20 mistakes)
group_4 = results_df[results_df['Mistakes'] == 4]['Cosine_Similarity']
group_20 = results_df[results_df['Mistakes'] == 20]['Cosine_Similarity']

mean_diff = group_4.mean() - group_20.mean()
pooled_std = np.sqrt(((len(group_4) - 1) * group_4.std()**2 + 
                      (len(group_20) - 1) * group_20.std()**2) / 
                     (len(group_4) + len(group_20) - 2))
cohens_d = mean_diff / pooled_std

print("=" * 80)
print("EFFECT SIZE ANALYSIS")
print("=" * 80)

print("\nCohen's d (4 mistakes vs 20 mistakes):")
print(f"  Mean similarity (4 mistakes): {group_4.mean():.4f}")
print(f"  Mean similarity (20 mistakes): {group_20.mean():.4f}")
print(f"  Difference: {mean_diff:.4f}")
print(f"  Cohen's d: {cohens_d:.4f}")

if abs(cohens_d) < 0.2:
    effect = "Negligible"
elif abs(cohens_d) < 0.5:
    effect = "Small"
elif abs(cohens_d) < 0.8:
    effect = "Medium"
else:
    effect = "Large"

print(f"  Effect size: {effect}")

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
ax.violinplot([group_4, group_20], positions=[1, 2], showmeans=True, showmedians=True)
ax.set_xticks([1, 2])
ax.set_xticklabels(['4 Mistakes', '20 Mistakes'])
ax.set_ylabel('Cosine Similarity', fontsize=12, fontweight='bold')
ax.set_title(f'Comparison of Extreme Groups\n(Cohen\'s d = {cohens_d:.4f})', 
             fontsize=14, fontweight='bold')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 7. Outlier Analysis

In [ ]:
# Identify and analyze outliers
Q1 = results_df['Cosine_Similarity'].quantile(0.25)
Q3 = results_df['Cosine_Similarity'].quantile(0.75)
IQR = Q3 - Q1

outliers = results_df[(results_df['Cosine_Similarity'] < Q1 - 1.5*IQR) | 
                      (results_df['Cosine_Similarity'] > Q3 + 1.5*IQR)]

print("=" * 80)
print("OUTLIER ANALYSIS")
print("=" * 80)
print(f"\nTotal observations: {len(results_df)}")
print(f"Number of outliers: {len(outliers)}")
print(f"Percentage: {len(outliers)/len(results_df)*100:.2f}%")

if len(outliers) > 0:
    print("\nOutlier sentences:")
    print(outliers[['Index', 'Mistakes', 'Cosine_Similarity']].to_string(index=False))
    
    # Visualize outliers
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.scatter(results_df['Index'], results_df['Cosine_Similarity'], 
               alpha=0.6, s=50, color='steelblue', label='Normal')
    ax.scatter(outliers['Index'], outliers['Cosine_Similarity'], 
               alpha=0.8, s=100, color='red', marker='X', label='Outliers', edgecolors='black', linewidth=1.5)
    ax.axhline(Q1 - 1.5*IQR, color='orange', linestyle='--', linewidth=1.5, label='Lower threshold')
    ax.axhline(Q3 + 1.5*IQR, color='orange', linestyle='--', linewidth=1.5, label='Upper threshold')
    ax.set_xlabel('Sentence Index', fontsize=12, fontweight='bold')
    ax.set_ylabel('Cosine Similarity', fontsize=12, fontweight='bold')
    ax.set_title('Outlier Detection in Similarity Scores', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 8. Normality Tests

In [ ]:
# Test normality assumption
from scipy.stats import shapiro, kstest

shapiro_stat, shapiro_p = shapiro(results_df['Cosine_Similarity'])
ks_stat, ks_p = kstest(results_df['Cosine_Similarity'], 'norm',
                       args=(results_df['Cosine_Similarity'].mean(), 
                             results_df['Cosine_Similarity'].std()))

print("=" * 80)
print("NORMALITY TESTS")
print("=" * 80)

print("\nShapiro-Wilk Test:")
print(f"  W-statistic: {shapiro_stat:.4f}")
print(f"  P-value: {shapiro_p:.6f}")
print(f"  Decision: {'Data appears normally distributed' if shapiro_p > 0.05 else 'Data may not be normally distributed'}")

# Q-Q plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Q-Q plot
stats.probplot(results_df['Cosine_Similarity'], dist="norm", plot=axes[0])
axes[0].set_title('Q-Q Plot for Normality', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)

# Histogram with normal curve
mu = results_df['Cosine_Similarity'].mean()
sigma = results_df['Cosine_Similarity'].std()
axes[1].hist(results_df['Cosine_Similarity'], bins=30, density=True, 
             alpha=0.7, color='steelblue', edgecolor='black')
x = np.linspace(results_df['Cosine_Similarity'].min(), results_df['Cosine_Similarity'].max(), 100)
axes[1].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label='Normal Distribution')
axes[1].set_xlabel('Cosine Similarity', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Density', fontsize=11, fontweight='bold')
axes[1].set_title('Histogram with Normal Curve Overlay', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Summary and Conclusions

In [ ]:
print("=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

print("\n1. CORRELATION:")
print(f"   - Pearson r = {pearson_r:.4f} (p = {pearson_p:.6f})")
print(f"   - Decision: {'Statistically significant' if pearson_p < 0.05 else 'Not significant'}")
print(f"   - Direction: {'NEGATIVE' if pearson_r < 0 else 'POSITIVE'}")
print(f"   - Strength: {'Weak' if abs(pearson_r) < 0.3 else 'Moderate' if abs(pearson_r) < 0.7 else 'Strong'}")

print("\n2. VARIANCE EXPLAINED:")
print(f"   - R-squared: {r_squared:.4f}")
print(f"   - {r_squared*100:.2f}% of variance in similarity explained by error count")

print("\n3. GROUP DIFFERENCES (ANOVA):")
print(f"   - F-statistic: {f_stat:.4f} (p = {p_value:.6f})")
print(f"   - Decision: {'Significant differences exist' if p_value < 0.05 else 'No significant differences'}")
print(f"   - Eta-squared: {eta_squared:.4f}")

print("\n4. EFFECT SIZE (4 vs 20 mistakes):")
print(f"   - Cohen's d: {cohens_d:.4f}")
print(f"   - Practical difference: {mean_diff:.4f}")

print("\n5. KEY FINDINGS:")
if pearson_p < 0.05 and pearson_r < 0:
    print("   ✓ There IS a statistically significant negative correlation")
    print("   ✓ More spelling errors lead to lower semantic similarity")
    print("   ✓ However, the effect is moderate (r ≈ -0.19)")
    print("   ✓ The translation pipeline shows reasonable robustness to errors")
else:
    print("   - No statistically significant relationship found")

print("\n" + "=" * 80)

## 10. Advanced Visualizations

In [ ]:
# Create comprehensive dashboard
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Main scatter plot with confidence interval
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax1.scatter(results_df['Mistakes'], results_df['Cosine_Similarity'], 
            alpha=0.5, s=60, color='steelblue', edgecolors='black', linewidth=0.5)
z = np.polyfit(results_df['Mistakes'], results_df['Cosine_Similarity'], 1)
p = np.poly1d(z)
x_trend = np.linspace(4, 20, 100)
y_trend = p(x_trend)
ax1.plot(x_trend, y_trend, 'r--', linewidth=2.5, alpha=0.8, label=f'Trend Line (r={pearson_r:.3f})')
ax1.set_xlabel('Number of Mistakes', fontsize=13, fontweight='bold')
ax1.set_ylabel('Cosine Similarity', fontsize=13, fontweight='bold')
ax1.set_title('Mistakes vs Similarity (All Data Points)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)

# 2. Distribution histogram
ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(results_df['Cosine_Similarity'], bins=25, edgecolor='black', alpha=0.7, color='lightcoral', orientation='horizontal')
ax2.set_ylabel('Cosine Similarity', fontsize=11, fontweight='bold')
ax2.set_xlabel('Frequency', fontsize=11, fontweight='bold')
ax2.set_title('Similarity Distribution', fontsize=12, fontweight='bold')
ax2.grid(alpha=0.3)

# 3. Box plot by error count
ax3 = fig.add_subplot(gs[1, 2])
bp = ax3.boxplot([results_df[results_df['Mistakes']==i]['Cosine_Similarity'].values 
                   for i in range(4, 21)], vert=False, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
ax3.set_yticklabels(range(4, 21))
ax3.set_xlabel('Cosine Similarity', fontsize=11, fontweight='bold')
ax3.set_ylabel('Mistakes', fontsize=11, fontweight='bold')
ax3.set_title('Distribution by Error Count', fontsize=12, fontweight='bold')
ax3.grid(alpha=0.3, axis='x')

# 4. Means with error bars
ax4 = fig.add_subplot(gs[2, :])
ax4.errorbar(averaged_df['Mistakes'], averaged_df['Avg_Similarity'], 
             yerr=averaged_df['Std_Dev'], fmt='o-', capsize=5, capthick=2,
             markersize=10, linewidth=2.5, color='darkgreen', ecolor='gray', label='Mean ± Std Dev')
ax4.fill_between(averaged_df['Mistakes'], 
                 averaged_df['Avg_Similarity'] - averaged_df['Std_Dev'],
                 averaged_df['Avg_Similarity'] + averaged_df['Std_Dev'],
                 alpha=0.2, color='green')
ax4.set_xlabel('Number of Mistakes', fontsize=13, fontweight='bold')
ax4.set_ylabel('Average Cosine Similarity', fontsize=13, fontweight='bold')
ax4.set_title('Average Similarity by Error Count with Confidence Bands', fontsize=14, fontweight='bold')
ax4.legend(fontsize=11)
ax4.grid(alpha=0.3)

plt.suptitle('Comprehensive Analysis Dashboard', fontsize=16, fontweight='bold', y=0.995)
plt.show()